In [ ]:
import pandas as pd, matplotlib.pyplot as plt; from matplotlib.backends.backend_pdf import PdfPages

# โหลดข้อมูลและแปลงค่าใน 2 บรรทัด
df = pd.read_csv('sales_transactions_cleaned.csv').assign(date=lambda x: pd.to_datetime(x['date']).dt.strftime('%Y-%m'))
df['revenue'] = (df['quantity'] * pd.to_numeric(df['price'].astype(str).str.replace('$', ''), errors='coerce')) - pd.to_numeric(df['discount_amount'], errors='coerce').fillna(0)

# คำนวณ 3 Metrics และหา Top 3
m_stats = df.groupby('date').agg(rev=('revenue','sum'), tx=('transaction_id','nunique')).assign(aov=lambda x: x['rev']/x['tx']).sort_index().reset_index()
t3 = m_stats.nlargest(3, 'rev')[['date', 'rev']].assign(rev=lambda x: x['rev'].apply(lambda v: f"${v:,.2f}"))

# วาด 3 กราฟเส้นและ 1 ตารางยัดลง PDF
with PdfPages('Session1_SalesTrendsv2.pdf') as pdf:
    fig, ax = plt.subplots(3, 1, figsize=(10, 12)); fig.tight_layout(pad=5.0)
    for i, (col, c, title) in enumerate([('rev','blue','Total Revenue'), ('tx','green','Transactions'), ('aov','red','Average Order Value')]):
        ax[i].plot(m_stats['date'], m_stats[col], marker='o', color=c); ax[i].set_title(title, weight='bold'); ax[i].tick_params(axis='x', rotation=45); ax[i].grid(ls='--')
    pdf.savefig(fig, bbox_inches='tight'); plt.close()
    
    fig, ax = plt.subplots(figsize=(6,2)); ax.axis('off'); tbl = ax.table(cellText=t3.values, colLabels=['Month','Total Revenue'], loc='center', cellLoc='center'); tbl.scale(1,2); ax.set_title('Top 3 Months', weight='bold')
    pdf.savefig(fig, bbox_inches='tight'); plt.close()

print("✅ สร้างไฟล์ข้อ 1.3 แบบย่อขั้นสุดเรียบร้อย!")